In [2]:
!pip install rdflib owlrl --quiet

In [3]:
from rdflib import Graph, Literal, Namespace, RDF, RDFS, OWL

# Define a tiny namespace and ontology
EX = Namespace("http://example.org/")

g = Graph()
g.bind("ex", EX)

# Ontology: define classes and a subclass relationship
g.add((EX.Person,    RDF.type, OWL.Class))
g.add((EX.Researcher, RDF.type, OWL.Class))
g.add((EX.Researcher, RDFS.subClassOf, EX.Person))  # every Researcher is a Person

# Data: Alice is a Researcher
g.add((EX.Alice, RDF.type, EX.Researcher))
g.add((EX.Alice, EX.knows, EX.Bob))

# Print as Turtle so you can see the RDF
print(g.serialize(format="turtle"))

@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Alice a ex:Researcher ;
    ex:knows ex:Bob .

ex:Person a owl:Class .

ex:Researcher a owl:Class ;
    rdfs:subClassOf ex:Person .




In [4]:
# Query: who is a Person? (Before reasoning, only explicit triples count.)
print("=== Before reasoning ===")
for row in g.query("PREFIX ex: <http://example.org/> SELECT ?p WHERE { ?p a ex:Person }"):
    print("  ", row[0])

# Now run OWL RL reasoning — derives that Alice is a Person because Researcher ⊑ Person
import owlrl
owlrl.DeductiveClosure(owlrl.OWLRL_Semantics).expand(g)

print("\n=== After reasoning ===")
for row in g.query("PREFIX ex: <http://example.org/> SELECT ?p WHERE { ?p a ex:Person }"):
    print("  ", row[0])
    

=== Before reasoning ===

=== After reasoning ===
   http://example.org/Alice
